In [1]:
import numpy as np

def compute_mae(var, var_hat):
    return np.sum(np.abs(var - var_hat)) / var.shape[0]

def compute_rmse(var, var_hat):
    return np.sqrt(np.sum((var - var_hat) ** 2) / var.shape[0])

In [2]:
import numpy as np

def ar(y, d):
    t = y.shape[0]
    vec = y[d:]
    mat = np.vstack([y[d-i-1 : t-i-1] for i in range(d)]).T   # (T-d) x d
    return np.linalg.pinv(mat) @ vec

def time_series_pred_multi(data, coef, horizon, step):
    t = data.shape[0]
    d = coef.shape[0]

    origins = np.arange(t - horizon, t) - step + 1             # (horizon,)
    assert origins.min() - d >= 0, "history too short"

    win = np.stack([data[o - d : o] for o in origins])         # horizon x d

    for _ in range(step):
        pred = win[:, ::-1] @ coef                             # (horizon,)
        win = np.hstack([win[:, 1:], pred[:, None]])

    return pred

In [3]:
import time
import pandas as pd
import numpy as np

data = pd.read_csv('../Wikipedia/top_pages.csv.gz', compression='gzip').iloc[:, 2 :].values # .iloc[: no, 2 :].values

step = 12
d = 7 * 24
horizon = 168
start = time.time()
est = np.zeros((data.shape[0], horizon))
for n in range(data.shape[0]):
    coef = ar(data[n, : 24 * (31 - 14)], d)
    est[n, :] = time_series_pred_multi(data[n, :], coef, horizon, step)
end = time.time()
print('Running time: %d seconds'%(end - start))
ground = data[:, 24 * 31 - horizon : 24 * 31]
print('MAE: {:.6}'.format(compute_mae(ground.reshape(-1), est.reshape(-1))))
print('RMSE: {:.6}'.format(compute_rmse(ground.reshape(-1), est.reshape(-1))))
print()

Running time: 53 seconds
MAE: 2645.8
RMSE: 1.60779e+05



In [4]:
import numpy as np

data = np.load('../NA-temperature/daymet_tmax_na_2010.npz')['arr_0']
for t in [11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]:
    data = np.append(data, np.load('../NA-temperature/daymet_tmax_na_20{}.npz'.format(t))['arr_0'], axis = 1)

In [5]:
import time
import numpy as np

step = 1
d = 30
horizon = 365
start = time.time()
est = np.zeros((data.shape[0], horizon))
for n in range(data.shape[0]):
    coef = ar(data[n, : - (365 + 365)], d)
    est[n, :] = time_series_pred_multi(data[n, :], coef, horizon, step)
end = time.time()
print('Running time: %d seconds'%(end - start))
ground = data[:, 24 * 31 - horizon : 24 * 31]
print('MAE: {:.6}'.format(compute_mae(ground.reshape(-1), est.reshape(-1))))
print('RMSE: {:.6}'.format(compute_rmse(ground.reshape(-1), est.reshape(-1))))
print()

Running time: 16 seconds
MAE: 5.44672
RMSE: 7.01205

